# Operation Safe-o-matic - a hands-on hardware assessment

You are on a hardware security team. A client has handed you a prototype access-control
module - the **Safe-o-matic 3000** - and asked a blunt question: *if someone has the board
on a bench, how far can they get?* You have the module, a ChipWhisperer-Husky, and three
hours. No source code, no keys, no password. Just the chip and its power pins.

You will work in two acts, from the crudest attack to the most surgical:

- **Act I - Fault injection.** The module decides *yes/no* in hardware. You will make it say
  *yes* when it should say *no*, by disturbing its power at the exact instant it decides.
- **Act II - Power analysis.** The module keeps secrets - a PIN and an AES key. You will read
  them back out of the current the chip draws while it uses them, without ever being told them.

Each **mission** below is one self-contained attack. The structure is always the same:

1. a short briefing on what the device does and where it is weak,
2. setup / build / flash cells you just **run**,
3. one or more cells **you write** (marked `# YOUR CODE` with a checklist of what it must do),
4. a **success check** the board itself confirms, and
5. **debrief questions** - write your answers in the markdown cell. Those are graded, not the
   screenshot. A working exploit you cannot explain is worth half a mark.

> This is the same bench and the same five targets you used in weeks 5, 6 and 8. The attacks
> are deliberately close to ones you have run before - the point today is to do them **without
> the finished code in front of you**, and to explain *why* each one works.

**What you hand in:** this notebook, run top to bottom, with your code, the success line the
board printed for each mission, and the debrief answers filled in.

## Bench setup (run, don't edit)

Two cells of plumbing used by every mission. The first locates your ChipWhisperer install so
the notebook runs from anywhere; the second is a reset helper for the SAM4S target (it needs a
**long** reset pulse - the short pulse from some older notebooks returns empty answers).

In [ ]:
PLATFORM = 'CW312_SAM4S'

import os, time, struct, random
import numpy as np
import chipwhisperer as cw

CWROOT = os.path.normpath(os.path.join(os.path.dirname(cw.__file__), "..", ".."))
SETUP  = os.path.join(CWROOT, "jupyter", "Setup_Scripts", "Setup_Generic.ipynb")
# drive-less, forward-slash root for the %%bash build cells (handles WSL /mnt/c, Git-Bash /c, *nix /)
CW_BASH = (CWROOT[0].lower() + CWROOT[2:].replace("\\", "/")) if os.name == "nt" else CWROOT.lstrip("/")
print("ChipWhisperer:", CWROOT)

In [ ]:
def reboot_flush():
    # the SAM4S wants a real, long reset pulse; 0.2 s leaves the serial buffer clean
    scope.io.nrst = "low"
    time.sleep(0.2)
    scope.io.nrst = "high_z"
    time.sleep(0.2)
    target.flush()

### The bench, at a glance (given - just run)

One diagram for the whole practice: what the Husky is wired to and which line each attack uses.
Act I rides the **clock / glitch** line into the chip; Act II listens on the **power-measure** line.
The serial line only carries your guesses and the board's short replies - the secrets never travel on it.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(11, 4.2)); ax.axis("off")
ax.set_xlim(0, 10); ax.set_ylim(1.5, 5.2)

def _box(x, y, w, h, label, fc):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.08", fc=fc, ec="0.3", lw=1.3))
    ax.text(x + w / 2, y + h / 2, label, ha="center", va="center", fontsize=10)

_box(0.4, 2.3, 3.0, 2.0, "ChipWhisperer-Husky\n(scope + glitch + USB-serial)", "#dce7f5")
_box(6.4, 2.3, 3.1, 2.0, "Safe-o-matic target\nCW312T-SAM4S", "#f5e7dc")

for y, lab, note, col in [
    (4.6, "clock / glitch  ->",     "Act I: the fault", "#c0392b"),
    (3.9, "power measure (ADC)  <-", "Act II: the leak", "#27ae60"),
    (3.2, "serial (commands)  <->",  "send guess / read reply", "0.35"),
    (2.6, "nRST + trigger  <->",     "reset and arm", "0.5"),
]:
    ax.add_patch(FancyArrowPatch((3.5, y), (6.3, y), arrowstyle="-", color=col, lw=2))
    ax.text(4.9, y + 0.1, lab, ha="center", fontsize=8.5, color=col)
    ax.text(4.9, y - 0.17, note, ha="center", fontsize=7.5, color="0.5", style="italic")
plt.title("One Husky, one target - a different line per act"); plt.tight_layout(); plt.show()

---
# Act I - Fault injection: corrupt what the chip computes

Deep inside the Safe-o-matic a routine counts: it runs a tight loop a fixed number of times to
build an internal value (think of an attempt counter, a checksum, or a key-stretch round count).
Get it to finish with the **wrong** number and you have proved you can derail the chip's control
flow from the outside - the same primitive that, aimed at the right instruction, skips a security
check entirely.

The weapon is a **clock glitch**: for a fraction of one cycle you insert an extra clock edge, and
the CPU mis-executes or skips the instruction straddling it. Three knobs place it:
- `scope.glitch.ext_offset` - **which** loop iteration region the glitch lands in,
- `scope.glitch.offset` / `scope.glitch.width` - **where inside** the target clock period the extra
  edge sits and how wide it is,
- `scope.glitch.repeat` - **how many** cycles in a row get glitched.

Every shot is sorted into three buckets: **normal** (the loop returns its expected total), **reset**
(we overdid it, the chip rebooted), **success** (any other total - the loop was derailed).

> Clock glitching the counting loop is the reliable, every-student-finishes attack. The harder
> variant - aiming a glitch at a one-shot security check so a wrong password is accepted - is in the
> stretch goal at the end; it uses the exact same loop, just a much narrower target in time.

## Mission 1 - derail the counting loop

**Build and flash the glitch firmware** (the simpleserial-glitch target from week 5). Its `'g'`
command runs a loop that counts to 2500 and returns the total.

In [ ]:
SS_VER_G = 'SS_VER_2_1'
FW_SRC_G = os.path.join(CWROOT, "firmware", "mcu", "simpleserial-glitch")

In [ ]:
%%bash -s "$PLATFORM" "$SS_VER_G" "$CW_BASH"
FW=""
for p in "/mnt/$3" "/$3"; do
  [ -d "$p/firmware/mcu/simpleserial-glitch" ] && FW="$p/firmware/mcu/simpleserial-glitch" && break
done
[ -z "$FW" ] && { echo "firmware dir not found for root: $3" >&2; exit 1; }
cd "$FW"
make PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 -j

In [ ]:
scope = cw.scope()
scope.default_setup()
target = cw.target(scope, cw.targets.SimpleSerial2)
prog = cw.programmers.SAM4SProgrammer
fw_path = os.path.join(FW_SRC_G, "simpleserial-glitch-{}.hex".format(PLATFORM))
cw.program_target(scope, prog, fw_path)
target.reset_comms()

def reboot_flush():
    scope.io.nrst = False;    time.sleep(0.05)
    scope.io.nrst = "high_z"; time.sleep(0.05); target.flush()
reboot_flush()

### Step 1.1 - baseline (given - just run)

With no glitch, the loop must return **2500**. Confirm that before attacking.

In [ ]:
target.simpleserial_write("g", bytearray([]))
v = target.simpleserial_read_witherrors('r', 4, glitch_timeout=10)
print("baseline count:", struct.unpack("<I", v['payload'])[0] if v['valid'] else v)   # expect 2500

### Step 1.2 - arm the clock glitch (given - just run)

`cglitch_setup()` wires the glitch module to XOR an extra edge into the target clock. On the Husky
it also disables the module, so you must re-enable it afterwards.

In [ ]:
scope.cglitch_setup()
if scope._is_husky:
    scope.glitch.enabled = True
cw.set_all_log_levels(cw.logging.CRITICAL)      # crashes flood warnings; silence them
scope.glitch.repeat = 20
scope.adc.lo_gain_errors_disabled = True
scope.adc.clip_errors_disabled = True
scope.adc.timeout = 0.1
STEPS = scope.glitch.phase_shift_steps
print("clock glitch armed | phase-shift steps per period:", STEPS)

### Step 1.2b - what the three knobs do (given - just run)

Before you sweep blindly, see the glitch. `clock_xor` inserts an **extra edge** into one clock
period: **offset** moves where that edge sits inside the period, **width** sizes it. The third knob,
**ext_offset** (not drawn here), picks *which* clock cycle gets this treatment - that is how you slide
the glitch onto the loop iteration you want to corrupt.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

t_ = np.arange(0, STEPS)
clock = (t_ < STEPS // 2).astype(int)                    # one period of the 50%-duty target clock
def glitched(width, offset):
    pulse = ((t_ >= offset) & (t_ < offset + width)).astype(int)
    return clock ^ pulse                                 # clock_xor: an extra edge where the pulse overlaps

fig, (a1, a2, a3) = plt.subplots(3, 1, figsize=(11, 6.5), sharex=True)
W0, O0 = 300, STEPS // 4
a1.step(t_, clock, where='post', color='0.5', lw=1.3, label='normal clock')
a1.step(t_, glitched(W0, O0) + 1.6, where='post', color='C1', lw=1.3, label=f'glitched (width={W0}, offset={O0})')
a1.axvspan(O0, O0 + W0, color='C1', alpha=0.15)
a1.set_title('clock_xor inserts an extra edge where the pulse overlaps the clock')
a1.legend(fontsize=8, loc='upper right'); a1.set_yticks([])
for k, off in enumerate([STEPS // 4, STEPS // 2, 3 * STEPS // 4]):
    a2.step(t_, glitched(300, off) + 1.8 * k, where='post', lw=1.2, label=f'offset={off}')
a2.set_title('offset moves WHERE the extra edge lands in the cycle (width=300)')
a2.legend(fontsize=8, loc='upper right'); a2.set_yticks([])
for k, w in enumerate([120, 350, 800]):
    a3.step(t_, glitched(w, STEPS // 2) + 1.8 * k, where='post', lw=1.2, label=f'width={w}')
a3.set_title('width sizes the inserted pulse (offset=%d)' % (STEPS // 2))
a3.legend(fontsize=8, loc='upper right'); a3.set_yticks([])
a3.set_xlabel('phase-shift step within one clock period (0 .. %d)' % STEPS)
plt.tight_layout(); plt.show()

### Step 1.3 - the campaign (you write the attack loop)

The core of Mission 1. Sweep `width`, `offset` and `ext_offset`, send `'g'` each time, and classify.

**Your loop must, for each `(width, offset, ext_offset)` in the grid:**
- reboot the target if it is mid-crash (`if scope.adc.state: reboot_flush()`),
- set `scope.glitch.width`, `scope.glitch.offset`, `scope.glitch.ext_offset`,
- `scope.arm()`, send `'g'`, `scope.capture()` (returns True if the chip crashed),
- read `'r'` 4 bytes and unpack a `<I`; classify: no/invalid answer -> **reset**; count `== 2500` ->
  **normal**; any other count -> **success** (print it and record the setting),
- after a reset, `reboot_flush()`.

Suggested grid: `width in range(0, STEPS//2, 200)`, `offset in range(0, STEPS, 200)`,
`ext_offset in range(0, 21)`. Shuffle each pass. Keep a tally and the winning settings.

In [ ]:
widths  = list(range(0, STEPS // 2, 200))
offsets = list(range(0, STEPS, 200))
exts    = list(range(0, 21))
grid    = [(w, o, e) for e in exts for o in offsets for w in widths]

tally = {"success": 0, "reset": 0, "normal": 0}
wins  = []
reboot_flush()
# YOUR CODE: sweep the grid (a few shuffled passes); for each (w, o, e):
#   if scope.adc.state: reboot_flush()
#   scope.glitch.width, scope.glitch.offset, scope.glitch.ext_offset = w, o, e
#   scope.arm(); target.simpleserial_write("g", bytearray([])); crashed = scope.capture()
#   read 'r' 4 bytes -> struct.unpack("<I", ...)[0]
#   classify: invalid/no answer -> "reset"; count==2500 -> "normal"; else -> "success"
#   on success: record (w, o, e, cnt) and print; on reset: reboot_flush()
raise NotImplementedError
print("success rate: %.2f%%" % (100 * tally['success'] / max(n, 1)))

### Step 1.4 - map where it worked (given - just run)

Plot the winning settings (offset vs ext_offset). A single hit is luck; a **cluster** is a result.

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(9, 4))
if wins:
    es = [e for (w, o, e, c) in wins]
    os_ = [o for (w, o, e, c) in wins]
    plt.scatter(es, os_, marker="+", s=90, color="tab:green")
    plt.title(f"{len(wins)} derailed runs - the sensitive window")
else:
    plt.title("no success yet - add passes or widen the grid")
plt.xlabel("ext_offset (which part of the loop)"); plt.ylabel("offset (edge position in the period)")
plt.tight_layout(); plt.show()
print("RESULT:", f"loop derailed at {len(wins)} setting(s)" if wins else "no success - keep sweeping")

### Mission 1 debrief (write your answers)

1. A "success" is any count that is not 2500. What are the physically most likely wrong counts, and
   what does each imply the glitch did to the loop (skipped an iteration? corrupted the counter?
   exited early)?
2. Your successes cluster at particular `offset` / `ext_offset` values. In one sentence, what is it
   about those settings that derails the loop while the rest run clean?
3. You saw three outcomes (normal / success / reset). Why does pushing the glitch harder (wider
   pulse) eventually turn successes into resets?

_Write here:_

1.
2.
3.

---
## Mission 2 - how reliable is your fault? (profiling)

A glitch you landed once is an anecdote. A real attacker needs the **hit rate**: if I fire this exact
setting over and over, how often does it work? Take your best winning setting from Mission 1 and
repeat it, counting the outcomes.

### Step 2.1 - pick the best point and repeat it (you write this)

**Your code must:** pick a `(width, offset, ext_offset)` from `wins` (the most common one is a good
choice), set the glitch to it, and fire `N = 100` shots of `'g'`, counting success / normal / reset.
Reboot after a reset. Print the counts and the **success rate**.

In [ ]:
from collections import Counter
assert wins, "run Mission 1 until you have at least one winning setting"
# YOUR CODE:
# best_w, best_o, best_e = most common (width, offset, ext_offset) in wins
# set scope.glitch.width / offset / ext_offset to it
# fire N=100 shots of 'g' at that fixed setting; count success / normal / reset
# print the counts and success rate
raise NotImplementedError

### Step 2.2 - the hit rate as a picture (given - just run)

Your 100 shots at one fixed setting, bucketed. A tall **success** bar means a dependable fault you
could script; a tall **reset** bar means you are pushing too hard and mostly crashing the chip.

In [ ]:
import matplotlib.pyplot as plt
counts = globals().get('c')
if counts:
    labels = ['success', 'normal', 'reset']
    vals = [counts.get(k, 0) for k in labels]
    plt.figure(figsize=(6, 3.6))
    bars = plt.bar(labels, vals, color=['tab:green', '0.6', 'tab:red'])
    for b, v in zip(bars, vals):
        plt.text(b.get_x() + b.get_width() / 2, v, str(v), ha='center', va='bottom', fontsize=10)
    plt.ylabel('shots (of %d)' % max(sum(vals), 1)); plt.title('Mission 2 - outcome of the repeated shots')
    plt.tight_layout(); plt.show()
    print('success rate: %.1f%%' % (100 * counts.get('success', 0) / max(sum(vals), 1)))
else:
    print('run Step 2.1 first - it fills the outcome counts this plot reads')

### Mission 2 debrief (write your answers)

1. Is your success rate at the fixed best point the same as the overall rate during the Mission 1
   sweep? Which is higher, and why?
2. A fault that works 2% of the time - is that good enough for a real attack on a device you can
   query repeatedly? What changes if each attempt costs a reboot and 2 seconds?
3. (If you have time) repeat this for a second winning setting. Does a point near the **edge** of the
   cluster behave differently from one at its centre?

_Write here:_

1.
2.
3.

---
# Act II - Power analysis: read the secrets off the power rail

Faulting the device forced a *decision*. It never told you a *secret*. Act II is the opposite
approach: leave the device running normally and **listen**. Every instruction the chip runs
moves charge, and the amount depends on the data it moves. Measure that current finely enough
and the secret falls out - the PIN in Mission 3, the AES key in Mission 4 - without a single
fault and without the secret ever appearing on the wire.

You will reconnect the board the normal way (no crowbar) and reuse the same capture bench.

## Mission 3 - Read the PIN with Simple Power Analysis (SPA)

The Safe-o-matic asks for a PIN and checks it one character at a time, bailing out at the
**first** wrong character. That early-exit is the whole weakness: a PIN that is correct for one
more character makes the check run one iteration longer, and that extra work is visible as extra
activity on the power trace. So you never guess the whole PIN - you peel it off **one character
at a time**, each correct character handed to you by the trace.

You do **not** know the PIN. You confirm success from the leak itself: the fully correct PIN makes
the device's check run one step longer than any wrong one.

**Build and flash the PIN firmware.** The instructor set a per-board PIN at build time; you are
given the alphabet and the length, nothing else.

In [ ]:
FW_SRC_SPA = os.path.join(CWROOT, "firmware", "mcu", "basic-passwdcheck")
SS_VER_SPA = 'SS_VER_2_1'

In [ ]:
%%bash -s "$PLATFORM" "$SS_VER_SPA" "$CW_BASH"
FW=""
for p in "/mnt/$3" "/$3"; do
  [ -d "$p/firmware/mcu/basic-passwdcheck" ] && FW="$p/firmware/mcu/basic-passwdcheck" && break
done
[ -z "$FW" ] && { echo "firmware dir not found for root: $3" >&2; exit 1; }
cd "$FW"
# PASSWD is set by the instructor per board; the default build keeps the stock PIN.
make PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2

In [ ]:
scope = cw.scope()
scope.default_setup()
target = cw.target(scope, cw.targets.SimpleSerial2)      # Setup_Generic builds this for SS_VER_2_1
prog = cw.programmers.SAM4SProgrammer
fw_path = os.path.join(FW_SRC_SPA, "basic-passwdcheck-{}.hex".format(PLATFORM))
cw.program_target(scope, prog, fw_path)
scope.adc.samples = 3000
print("flashed; samples =", scope.adc.samples)

### Why this leaks, before you capture (given - just run)

The attack in one picture. The check compares your guess to the PIN character by character and
**stops at the first mismatch**. The more leading characters you get right, the more iterations it
runs before bailing - and more iterations means a longer stretch of visible activity on the power
trace. That staircase is the whole attack: at each position, the right character is the one that
buys you one more step of work.

In [ ]:
import matplotlib.pyplot as plt
correct = np.arange(0, 6)          # how many leading characters match (0..5 for a 5-char PIN)
work = correct + 1                 # compare runs one iteration past the last match, then bails
plt.figure(figsize=(9, 3.6))
cols = ['0.6'] * 5 + ['tab:green']
plt.bar(correct, work, color=cols)
plt.text(5, work[-1] + 0.05, 'full PIN -> runs to the end', ha='center', color='tab:green', fontsize=9)
plt.xlabel('leading characters guessed correctly')
plt.ylabel('compare iterations\n(= visible work on the trace)')
plt.title('early-exit check: each correct character buys one more iteration of visible work')
plt.tight_layout(); plt.show()

### Step 3.1 - capture one PIN-attempt trace

The target prints a boot banner, then waits for the PIN. You must reset it, drain the banner so
the trace is only the check, send one guess, and capture. Write `cap_pass_trace(guess)`.

**Your function must:**
- pull `scope.io.nrst` low ~0.05 s then release to `"high_z"` and wait ~0.15 s for the banner,
- drain the serial with `target.in_waiting()` / `target.read()` until empty,
- `scope.arm()`, `target.write(guess)`, `scope.capture()`,
- return `scope.get_last_trace()`.

Also write `avg_trace(guess, n=4)` returning the mean of `n` captures - the per-character leak is
small, so averaging beats the noise.

In [ ]:
def cap_pass_trace(pass_guess):
    # YOUR CODE:
    # 1. reset the target: nrst "low" ~0.05s, then "high_z" ~0.15s (let it boot + print banner)
    # 2. drain the banner: while target.in_waiting() > 0: target.read(...)
    # 3. scope.arm(); target.write(pass_guess); scope.capture()
    # 4. return scope.get_last_trace()
    raise NotImplementedError

def avg_trace(pass_guess, n=4):
    # YOUR CODE: return the mean of n cap_pass_trace(pass_guess) captures (np.mean(..., axis=0))
    raise NotImplementedError

t = cap_pass_trace("a\n")
print("trace length:", len(t), "| min/max: %.3f / %.3f" % (t.min(), t.max()))

### Step 3.2 - see the leak (given - just run)

Pick any character that is certainly wrong as a **reference**, then compare a candidate against it.
A correct first character makes the trace diverge from the reference; a wrong one tracks it. This
cell only *visualises* the effect across the whole alphabet - your recovery code comes next.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

CHARS = 'abcdefghijklmnopqrstuvwxyz0123456789'
W = 1500
ref = avg_trace("\x01\n")                       # 0x01 is never in the PIN -> a guaranteed-wrong reference

plt.figure(figsize=(13, 4))
best_c, best_d = None, -1
for c in CHARS:
    d_ = np.abs(avg_trace(c + "\n", 2) - ref)
    peak = d_[:W].max()
    if peak > best_d: best_d, best_c = peak, c
    plt.plot(d_[:W], color="0.6", lw=0.5, alpha=0.5)
# redraw the current leader on top
d_ = np.abs(avg_trace(best_c + "\n", 2) - ref)
plt.plot(d_[:W], color="tab:green", lw=1.6, label=f"'{best_c}' (largest divergence)")
plt.xlabel("ADC sample"); plt.ylabel("|guess - ref|")
plt.title("36 first-character guesses vs a wrong reference - one erupts, the rest stay flat")
plt.legend(); plt.tight_layout(); plt.show()
print("loudest first character looks like:", repr(best_c))

### Step 3.3 - turn the picture into a recovery (you write this)

Score each candidate with the **sum of absolute differences (SAD)** from a wrong reference, pick
the loudest, then grow a known prefix and attack the next position. Write `recover_char(prefix)`
and the loop over all positions.

**`recover_char(prefix)` must:** take a reference `cap_pass_trace(prefix + "\x01\n")`, then for each
`c` in `CHARS` score `sum(abs(cap_pass_trace(prefix + c + "\n") - ref))`, and return the highest-scoring `c`.

**The loop must:** start `guessed = ""`, and for each position append `recover_char(guessed)` until
the PIN length is reached (try 5; raise it if the board does not grant). Print the PIN as it grows.

In [ ]:
def recover_char(prefix):
    # YOUR CODE:
    # ref = cap_pass_trace(prefix + "\x01\n")        # 0x01 never appears in the PIN
    # for each c in CHARS: score[c] = sum(abs(cap_pass_trace(prefix + c + "\n") - ref))
    # return the c with the HIGHEST score
    raise NotImplementedError

PIN_LEN = 5          # the length you were given (raise if the board does not grant)
guessed = ""
for position in range(PIN_LEN):
    # YOUR CODE: c = recover_char(guessed); guessed += c; print progress
    raise NotImplementedError
print("\nRecovered PIN:", repr(guessed))

### Step 3.4 - prove the PIN is complete (given - just run)

How do you know the recovered PIN is right without being told? Use the leak itself. A **fully
correct** PIN makes the device's check run its compare loop all the way to the end; a PIN with any
wrong character bails out one iteration sooner. So the correct PIN's trace diverges from a wrong
reference **furthest in time** and has the largest total difference. This cell captures your
recovered PIN and a variant with the last character flipped, and shows the correct one runs longer -
that extra iteration is the device confirming, in power, that every character matched.

(The device also lights its "access granted" LED here; the power proof below is what you hand in.)

In [ ]:
ref   = avg_trace("\x01\n")                              # guaranteed-wrong reference
full  = avg_trace(guessed + "\n")                        # your recovered PIN
alt   = 'a' if guessed[-1] != 'a' else 'b'
wrong = avg_trace(guessed[:-1] + alt + "\n")             # same PIN, last character wrong

sad_full  = float(np.sum(np.abs(full  - ref)))
sad_wrong = float(np.sum(np.abs(wrong - ref)))

plt.figure(figsize=(13, 3.8))
plt.plot(np.abs(full  - ref)[:W], color="tab:green", lw=1.0, label=f"recovered PIN {guessed!r} (runs full loop)")
plt.plot(np.abs(wrong - ref)[:W], color="tab:red",   lw=0.9, alpha=0.8, label="last character wrong (bails out early)")
plt.xlabel("ADC sample"); plt.ylabel("|trace - reference|")
plt.title("the correct PIN's divergence extends furthest - every character matched")
plt.legend(); plt.tight_layout(); plt.show()

print(f"SAD(correct) = {sad_full:.0f}   SAD(last-char-wrong) = {sad_wrong:.0f}")
print("RESULT:", "PIN CONFIRMED - correct PIN runs the loop longest" if sad_full > sad_wrong
      else "inconclusive - average more traces (raise n) or re-check the recovery")
scope.dis(); target.dis()

### Mission 3 debrief (write your answers)

1. The PIN never left the chip and nothing was faulted. **What** exactly leaks the characters -
   name the feature of the check that makes SPA work here.
2. You used `0x01` as the reference character. Why does the choice of reference not matter, as long
   as it is wrong?
3. This attack costs about `alphabet x length` traces. How would a **constant-time** compare
   (always looping over the full PIN, no early exit) change that - and what would you then have to
   attack instead? *(You get to try exactly that in the bonus.)*

_Write here:_

1.
2.
3.

---
## Mission 4 - Lift the AES key with Correlation Power Analysis (CPA)

The PIN gate fell to timing you could see by eye. The AES key will not: AES does the same
operations whatever the key, so the leak is **data-dependent**, not **time-dependent**, and it is
buried under noise. CPA digs it out statistically. For each of the 16 key bytes independently, you
guess all 256 values, predict the power each guess implies via `HW(Sbox(plaintext XOR guess))`,
and keep the guess whose prediction **correlates** best with the measured power. Divide and conquer:
16 small 256-way problems instead of one impossible 2^128 one.

The board holds a key you were **not** told. You will confirm the recovered key by checking it
reproduces the target's own ciphertext.

### The model, before you capture (given - just run)

CPA in one diagram. You know the plaintext; you guess one key byte; you predict how much power the
chip burns for that guess (`HW(Sbox(pt XOR guess))`); you correlate that prediction against the
measured power across many traces. The right guess tracks the measurements; the 255 wrong ones do
not. Then repeat for all 16 bytes - 16 small 256-way problems, not one impossible 2^128 one.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(11, 3.4)); ax.axis("off"); ax.set_xlim(0, 14); ax.set_ylim(0, 3)
chain = [("known\nplaintext byte", "#dce7f5"), ("XOR\nkey guess g", "#f5e7dc"),
         ("S-box", "#f5e7dc"), ("Hamming\nweight", "#f5e7dc"), ("predicted\npower", "#e7f5dc")]
x = 0.3
for lab, fc in chain:
    if x > 0.3:
        ax.add_patch(FancyArrowPatch((x - 0.3, 1.6), (x, 1.6), arrowstyle="-|>", mutation_scale=12, color="0.4"))
    ax.add_patch(FancyBboxPatch((x, 1.1), 1.8, 1.0, boxstyle="round,pad=0.06", fc=fc, ec="0.3", lw=1.2))
    ax.text(x + 0.9, 1.6, lab, ha="center", va="center", fontsize=8.5)
    x += 2.1
ax.add_patch(FancyArrowPatch((x - 0.3, 1.6), (x, 1.6), arrowstyle="-|>", mutation_scale=12, color="0.4"))
ax.add_patch(FancyBboxPatch((x, 1.1), 2.7, 1.0, boxstyle="round,pad=0.06", fc="#f0d9e6", ec="0.3", lw=1.2))
ax.text(x + 1.35, 1.6, "correlate vs\nmeasured power", ha="center", va="center", fontsize=8.5)
ax.text(7, 0.45, "do this for all 256 guesses -> the one that correlates is the key byte -> repeat x16",
        ha="center", fontsize=9, color="0.3", style="italic")
plt.title("CPA: predict the leak for each guess, keep the guess that matches"); plt.tight_layout(); plt.show()

**Build and flash the AES firmware.**

In [ ]:
CRYPTO_TARGET = 'TINYAES128C'
SS_VER_AES = 'SS_VER_2_1'
FW_SRC_AES = os.path.join(CWROOT, "firmware", "mcu", "simpleserial-aes")

In [ ]:
%%bash -s "$PLATFORM" "$CRYPTO_TARGET" "$SS_VER_AES" "$CW_BASH"
FW=""
for p in "/mnt/$4" "/$4"; do
  [ -d "$p/firmware/mcu/simpleserial-aes" ] && FW="$p/firmware/mcu/simpleserial-aes" && break
done
[ -z "$FW" ] && { echo "firmware dir not found for root: $4" >&2; exit 1; }
cd "$FW"
make PLATFORM=$1 CRYPTO_TARGET=$2 SS_VER=$3 -j

In [ ]:
scope = cw.scope()
scope.default_setup()
target = cw.target(scope, cw.targets.SimpleSerial2)
prog = cw.programmers.SAM4SProgrammer
fw_path = os.path.join(FW_SRC_AES, "simpleserial-aes-{}.hex".format(PLATFORM))
cw.program_target(scope, prog, fw_path)
target.reset_comms()

def reboot_flush():
    scope.io.nrst = "low";  time.sleep(0.2)
    scope.io.nrst = "high_z"; time.sleep(0.2); target.flush()
reboot_flush()

### Step 4.1 - load a key you will not look at (given - just run)

The instructor-style setup: a random key is loaded into the target and **not printed**. Your job
is to recover it from power alone; the check at the end matches the target ciphertext, never the
key value, so you never peek.

In [ ]:
import secrets
HIDDEN_KEY = bytearray(secrets.token_bytes(16))   # loaded into the board; treated as unknown
target.set_key(HIDDEN_KEY)
del HIDDEN_KEY                                     # gone from view - recover it, don't read it
print("a random key has been loaded into the target (not shown)")

### Step 4.2 - capture N traces (you write this)

Each trace: a fresh random 16-byte plaintext, encrypted under the loaded key. Store the trace and
the plaintext (CPA needs the known input). Write the capture loop.

**It must:** set `scope.adc.samples = 5000`, `scope.gain.db = 25`; for `N_TRACES` iterations make a
random plaintext (`ktp = cw.ktp.Basic(); _, text = ktp.next()`), `scope.arm()`,
`target.simpleserial_write('p', text)`, `scope.capture()`, read `'r'` 16 bytes, and append the
trace and `list(text)`. Build `traces` (float array) and `textins` (uint8 array).

In [ ]:
scope.adc.samples = 5000
scope.gain.db = 25
ktp = cw.ktp.Basic()

N_TRACES = 500          # the SAM4S leaks weakly; a few hundred traces is realistic
traces, textins = [], []
# YOUR CODE: loop until you have N_TRACES:
#   _, text = ktp.next()                       # fresh random plaintext
#   scope.arm(); target.simpleserial_write('p', text)
#   if scope.capture(): reboot_flush(); continue
#   ct = target.simpleserial_read('r', 16)
#   append np.array(scope.get_last_trace()) and list(text)
raise NotImplementedError
traces = np.array(traces)
textins = np.array(textins, dtype=np.uint8)
print("captured", traces.shape)

### Step 4.3 - the CPA engine (you write the correlation)

The S-box and Hamming-weight tables are given. Write `cpa_byte(traces, pts, byte)` returning the
absolute Pearson correlation for every one of the 256 guesses at every sample.

**Recipe:** centre each sample column (`traces - traces.mean(axis=0)`); for each guess build the
hypothesis `h = HW[SBOX[pts[:, byte] XOR guess]]` (shape 256 x N) and centre it; then
`r = (h_centred @ t_centred) / (norm_h * norm_t)`. Return `|r|` (use `np.nan_to_num`).

In [ ]:
SBOX = np.array([
    0x63,0x7c,0x77,0x7b,0xf2,0x6b,0x6f,0xc5,0x30,0x01,0x67,0x2b,0xfe,0xd7,0xab,0x76,
    0xca,0x82,0xc9,0x7d,0xfa,0x59,0x47,0xf0,0xad,0xd4,0xa2,0xaf,0x9c,0xa4,0x72,0xc0,
    0xb7,0xfd,0x93,0x26,0x36,0x3f,0xf7,0xcc,0x34,0xa5,0xe5,0xf1,0x71,0xd8,0x31,0x15,
    0x04,0xc7,0x23,0xc3,0x18,0x96,0x05,0x9a,0x07,0x12,0x80,0xe2,0xeb,0x27,0xb2,0x75,
    0x09,0x83,0x2c,0x1a,0x1b,0x6e,0x5a,0xa0,0x52,0x3b,0xd6,0xb3,0x29,0xe3,0x2f,0x84,
    0x53,0xd1,0x00,0xed,0x20,0xfc,0xb1,0x5b,0x6a,0xcb,0xbe,0x39,0x4a,0x4c,0x58,0xcf,
    0xd0,0xef,0xaa,0xfb,0x43,0x4d,0x33,0x85,0x45,0xf9,0x02,0x7f,0x50,0x3c,0x9f,0xa8,
    0x51,0xa3,0x40,0x8f,0x92,0x9d,0x38,0xf5,0xbc,0xb6,0xda,0x21,0x10,0xff,0xf3,0xd2,
    0xcd,0x0c,0x13,0xec,0x5f,0x97,0x44,0x17,0xc4,0xa7,0x7e,0x3d,0x64,0x5d,0x19,0x73,
    0x60,0x81,0x4f,0xdc,0x22,0x2a,0x90,0x88,0x46,0xee,0xb8,0x14,0xde,0x5e,0x0b,0xdb,
    0xe0,0x32,0x3a,0x0a,0x49,0x06,0x24,0x5c,0xc2,0xd3,0xac,0x62,0x91,0x95,0xe4,0x79,
    0xe7,0xc8,0x37,0x6d,0x8d,0xd5,0x4e,0xa9,0x6c,0x56,0xf4,0xea,0x65,0x7a,0xae,0x08,
    0xba,0x78,0x25,0x2e,0x1c,0xa6,0xb4,0xc6,0xe8,0xdd,0x74,0x1f,0x4b,0xbd,0x8b,0x8a,
    0x70,0x3e,0xb5,0x66,0x48,0x03,0xf6,0x0e,0x61,0x35,0x57,0xb9,0x86,0xc1,0x1d,0x9e,
    0xe1,0xf8,0x98,0x11,0x69,0xd9,0x8e,0x94,0x9b,0x1e,0x87,0xe9,0xce,0x55,0x28,0xdf,
    0x8c,0xa1,0x89,0x0d,0xbf,0xe6,0x42,0x68,0x41,0x99,0x2d,0x0f,0xb0,0x54,0xbb,0x16,
], dtype=np.uint8)
HW = np.array([bin(x).count("1") for x in range(256)], dtype=np.float64)

In [ ]:
def cpa_byte(traces, pts, byte):
    # YOUR CODE:
    # t  = traces - column means ; tn = sqrt(sum t^2 over traces)
    # h  = HW[SBOX[pts[:, byte] XOR guess]] for all 256 guesses (shape 256 x N); centre it; hn = row norms
    # r  = (h @ t) / (hn * tn)        # Pearson r, 256 x samples
    # return np.nan_to_num(np.abs(r))
    raise NotImplementedError

r = cpa_byte(traces, textins, 0)
best = int(np.argmax(r.max(axis=1)))
print(f"byte 0 best guess = 0x{best:02X}  |r| = {r.max():.3f}")

### Step 4.3b - see the right byte spike (given - just run)

Your `cpa_byte` output for byte 0, drawn two ways. Left: the correlation over time for all 256
guesses - 255 stay in the noise, one rises at the moment the S-box output is computed. Right: the
peak correlation per guess - the single bar standing clear of the pack is the key byte.

In [ ]:
import matplotlib.pyplot as plt
rr = globals().get('r')
if rr is not None:
    peaks = rr.max(axis=1)
    win = int(np.argmax(peaks))
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 3.8))
    a1.plot(rr.T, color='0.75', lw=0.4)
    a1.plot(rr[win], color='tab:red', lw=1.3, label=f'guess 0x{win:02X}')
    a1.set_xlabel('ADC sample'); a1.set_ylabel('|correlation|')
    a1.set_title('all 256 guesses over time'); a1.legend(fontsize=8)
    a2.bar(range(256), peaks, color='0.7', width=1.0)
    a2.bar([win], [peaks[win]], color='tab:red', width=2.5)
    a2.set_xlabel('key-byte guess (0..255)'); a2.set_ylabel('peak |correlation|')
    a2.set_title(f'byte 0: 0x{win:02X} stands clear')
    plt.tight_layout(); plt.show()
    print(f"byte 0 best 0x{win:02X} | peak |r|={peaks[win]:.3f} | next best {sorted(peaks)[-2]:.3f}")
else:
    print('write cpa_byte in Step 4.3 first, then run this cell')

### Step 4.4 - all 16 bytes, then confirm against the device (you write the sweep)

Byte 0 is not special. Recover every byte, assemble the key, and **prove it** by encrypting a fresh
plaintext in software under your recovered key and checking it equals what the board returns.

**Your code must:** loop `b` in `0..15` taking `argmax` of `cpa_byte(...).max(axis=1)` into `rec`;
then send a random plaintext with `'p'`, read the `'r'` ciphertext, and compare to
`AES.new(bytes(rec), AES.MODE_ECB).encrypt(bytes(text))`.

In [ ]:
from Crypto.Cipher import AES
# YOUR CODE:
# rec = bytearray of argmax(cpa_byte(traces, textins, b).max(axis=1)) for b in 0..15
# send a fresh plaintext, read the target ciphertext ct
# ref = AES.new(bytes(rec), AES.MODE_ECB).encrypt(bytes(text))
# success = (ct == ref)
raise NotImplementedError

### Mission 4 debrief (write your answers)

1. Why attack `Sbox(pt XOR key)` and not `pt XOR key` directly? (Mission 5 lets you *try* the wrong
   one.)
2. You recovered 16 bytes independently. Why does that not reduce AES security to 16 x 8 bits of
   brute force in general - what did the power trace give you that a black-box attacker lacks?
3. Which byte, if any, needed the most traces to settle? Look at the `|r|` of the right guess vs
   the best wrong guess - what makes some bytes harder?

_Write here:_

1.
2.
3.

---
## Mission 5 (bonus) - why the model matters: attack the wrong intermediate

Reuse the Mission 4 traces - no new capture. Re-run the attack with the hypothesis
`HW(pt XOR key)` instead of `HW(Sbox(pt XOR key))`: drop the S-box. Then explain what you see.

In [ ]:
def cpa_byte_nosbox(traces, pts, byte):
    # YOUR CODE: same as cpa_byte but hypothesis h = HW[ pts[:, byte] XOR guess ]  (NO SBOX[...])
    raise NotImplementedError

p = cpa_byte_nosbox(traces, textins, 0).max(axis=1)
order = np.argsort(p)[::-1]
print("top 5 guesses (no S-box):", [f"0x{g:02X}:{p[g]:.3f}" for g in order[:5]])

### Step 5.1 - right model vs wrong model (given - just run)

Peak correlation per guess, both ways. With the S-box (left) one bar stands alone; without it
(right) a cluster of near-miss guesses rises together and the winner is ambiguous. Same traces,
same math - only the predicted intermediate changed.

In [ ]:
import matplotlib.pyplot as plt
if 'cpa_byte' in globals() and 'cpa_byte_nosbox' in globals() and 'traces' in globals():
    pk_s  = cpa_byte(traces, textins, 0).max(axis=1)
    pk_ns = cpa_byte_nosbox(traces, textins, 0).max(axis=1)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 3.6), sharey=True)
    for ax, pk, ttl in [(a1, pk_s, 'with S-box (correct model)'), (a2, pk_ns, 'no S-box (wrong model)')]:
        w = int(np.argmax(pk))
        ax.bar(range(256), pk, color='0.7', width=1.0)
        ax.bar([w], [pk[w]], color='tab:red', width=2.5)
        ax.set_title('%s\ntop-to-2nd gap = %.3f' % (ttl, pk[w] - sorted(pk)[-2]))
        ax.set_xlabel('key-byte guess')
    a1.set_ylabel('peak |correlation|'); plt.tight_layout(); plt.show()
else:
    print('run Step 4.3 (cpa_byte) and the Mission 5 code above first')

### Mission 5 debrief

1. Several guesses now peak together instead of one clear winner. Why does dropping the S-box make
   near-miss key guesses look almost as good as the right one? (Hint: the S-box is highly
   non-linear; `pt XOR key` is not.)
2. In one sentence: what property must a good CPA target point have?

_Write here:_

1.
2.

---
# Operation debrief

You took one board from *no information* to *full compromise*:

- **Act I** derailed the chip's control flow with a power fault - no secret needed, just timing.
- **Act II** read the PIN and then the AES key out of the power the chip drew - no fault needed,
  just statistics.

Hand in this notebook with every mission's **success / RESULT line** and your **debrief answers**.
The grade is on the explanations: an exploit you can run is a start, an exploit you can explain is
the skill.

**If you finished early:** two stretch goals below.

## Stretch goal A - force the authentication gate (harder glitch)

Same firmware, same clock glitch, a much smaller target in time. The `'p'` command checks a 5-byte
password and returns `passok` (1 = granted, 0 = denied). Send the **wrong** password `b"toukl"` and
glitch the single `passok = 0` store so a wrong password is accepted (`passok == 1`).

Reuse your Mission 1 campaign, but drive `'p'` with `WRONG = b"toukl"` and classify `passok == 1` as
success. This store is one instruction, not a 2500-iteration loop, so the sensitive window is tiny -
expect to sweep much longer, and do not be surprised if some boards need many thousands of shots or
a different `repeat`. That difficulty *is* the lesson: a one-shot check is far harder to fault than a
long loop.

- How many shots did a grant take compared with derailing the loop? Why the difference?

## Stretch goal B - break your own countermeasure

Rewrite the PIN compare so it is **constant-time**: loop over the whole PIN every time, accumulate a
difference, and never `break` early. Rebuild, reflash, and rerun your Mission 3 SPA attack.

- Does the SPA attack still read the PIN character by character? Why not?
- Look at the power trace of your constant-time version: is `result |= (a ^ b)` really constant-time
  on this chip, or can you still see something? Judge from the **trace**, not the source.